In [3]:
import os
import pandas as pd


In [4]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps\\research'

In [5]:
os.chdir(r"C:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps")


In [6]:
%pwd

'C:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps'

In [75]:
import numpy as np

In [7]:
from dataclasses import dataclass
from pathlib import Path
@dataclass

class model_evalulation_path:
    sim_matrix_path: Path
    featured_df_path :Path 
    model_eval_metrics : Path


In [8]:
from src.recommendation_system.constants import CONFIG_PATH
from src.recommendation_system.utils.common import read_yaml , create_dir
from dataclasses import dataclass
from src.recommendation_system.logging import logger
import pickle

In [9]:
class Congif_manager:

    def __init__(self, config =CONFIG_PATH):
        self.config = read_yaml(config)

        create_dir([self.config.artifacts_root])

    def get_model_evaluation(self) -> model_evalulation_path:
        config = self.config.model_evalvation

        create_dir([config.model_eval_metrics])

        return model_evalulation_path(
            sim_matrix_path        = (config.sim_matrix_path),
            featured_df_path       = (config.featured_df_path),
            model_eval_metrics = (config.model_eval_metrics)
            #mlflow_uri             = config.mlflow_uri,
            #mlflow_experiment      = config.mlflow_experiment
        ) 

In [78]:
class ModelEvaluation:

    def __init__(self, config: model_evalulation_path):
        self.config     = config
        self.sim_matrix = None
        self.df         = None

    # ------------------------------------------------------------------ #
    #  Step 1 — Load                                                      #
    # ------------------------------------------------------------------ #
    def load_artifacts(self):
        logger.info("[ Step 1 ] Loading artifacts")

        with open(self.config.sim_matrix_path, 'rb') as f:
            self.sim_matrix = pickle.load(f)

        with open(self.config.featured_df_path, 'rb') as f:
            self.df = pickle.load(f)

        logger.info("  sim_matrix : %s", str(self.sim_matrix.shape))
        logger.info("  df         : %s", str(self.df.shape))
        logger.info("[ Step 1 ] Done\n")
    
    def get_test_sample(self) -> pd.DataFrame:
        logger.info("[ Step 2 ] Building test sample")

        stratified = (
            self.df.groupby("aesthetic", group_keys=False)
                   .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
        )
        top_rated    = self.df.nlargest(10, "rating")
        bottom_rated = self.df.nsmallest(10, "rating")

        test_set = (
            pd.concat([stratified, top_rated, bottom_rated])
              .drop_duplicates()
              .reset_index(drop=True)
        )

        logger.info("  Sample size : %d", len(test_set))
        logger.info("[ Step 2 ] Done\n")
        
        return test_set
    
    def precession(self, revelant , totla, topk):

        revelant = set(revelant[::topk])

        total = set(totla)

        return revelant/total
    

    def precision_at_k(self, recommended_idx, relevant_idx, k) -> float:

        recommended_set = set(recommended_idx[:k])   # top k as set
        relevant_set    = set(relevant_idx)           # ground truth as set
        hits            = len(recommended_set & relevant_set)  # intersection
        return hits / k

    # ------------------------------------------------------------------ #
    #  Step 4 — NDCG@K                                                    #
    # ------------------------------------------------------------------ #
    def ndcg_at_k(self, recommended_idx, relevant_idx, k) -> float:

        relevant_set = set(relevant_idx)

        # actual ranking score
        dcg = sum(
            1 / np.log2(i + 2)
            for i, idx in enumerate(recommended_idx[:k])
            if idx in relevant_set
        )

        # best possible score
        ideal_hits = min(k, len(relevant_set))
        idcg       = sum(1 / np.log2(i + 2) for i in range(ideal_hits))

        return dcg / idcg if idcg > 0 else 0.0

    # ------------------------------------------------------------------ #
    #  Step 5 — Score One Product                                         #
    # ------------------------------------------------------------------ #
    def score_product(self, row, top_k=10) -> dict:

        product_id = row.name
        aesthetic  = row["aesthetic"]

        # ground truth from featured_df
        relevant_idx = self.df[
            (self.df["aesthetic"] == aesthetic) &
            (self.df.index        != product_id)
        ].index.tolist()

        # engine 1 from sim_matrix
        scores             = self.sim_matrix[product_id].copy()
        scores[product_id] = -1
        top_idx            = scores.argsort()[::-1][0:top_k]

        # calculate metrics
        precision = self.precision_at_k(top_idx, relevant_idx, top_k)
        ndcg      = self.ndcg_at_k(top_idx,      relevant_idx, top_k)

        logger.info(
            "  %-35s precision=%.2f  ndcg=%.2f",
            row["product_name"][:35],
            precision,
            ndcg
        )

        return {
            "product_name"   : row["product_name"],
            "aesthetic"      : aesthetic,
            "n_relevant"     : len(relevant_idx),
            "precision_at_k" : round(precision, 4),
            "ndcg_at_k"      : round(ndcg,      4),
        }
    

    def run_evaluation(self, top_k=10):
        logger.info("===== Evaluation STARTED =====")

        if self.df is None or self.sim_matrix is None:
            self.load_artifacts()

        test_sample = self.get_test_sample()

        logger.info("[ Step 3 ] Scoring %d products", len(test_sample))

        records = []
        for _, row in test_sample.iterrows():
            result = self.score_product(row, top_k=top_k)
            records.append(result)

        metrics_df = pd.DataFrame(records)
        logger.info("[ Step 3 ] Done\n")

        # average all 70 scores
        summary = {
            "n_products_tested" : len(metrics_df),
            "top_k"             : top_k,
            "precision_at_k"    : round(metrics_df["precision_at_k"].mean(), 4),
            "ndcg_at_k"         : round(metrics_df["ndcg_at_k"].mean(),      4),
        }

        logger.info("======= SUMMARY =======")
        for k, v in summary.items():
            logger.info("  %-25s : %s", k, v)

        # save
        out = self.config.model_eval_metrics
        metrics_df.to_csv(out / "eval_metrics.csv", index=False)
        

        logger.info("===== Evaluation FINISHED =====\n")
        return summary, metrics_df


In [79]:
con   = Congif_manager()
model = con.get_model_evaluation()
model = ModelEvaluation(model)
model.run_evaluation()   

test_sample = model.get_test_sample()
print("shape            :", test_sample.shape)
print("aesthetic counts :")
print(test_sample["aesthetic"].value_counts())

[2026-03-04 00:18:37,197: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-04 00:18:37,202: INFO: common: created directory at: artifacts]
[2026-03-04 00:18:37,203: INFO: common: created directory at: artifacts/model_evalvation/model/]
[2026-03-04 00:18:37,205: INFO: 376599490: ===== Evaluation STARTED =====]
[2026-03-04 00:18:37,207: INFO: 376599490: [ Step 1 ] Loading artifacts]
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_7912\1399911550.py", line 4, in <module>
    model.run_evaluation()
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_7912\376599490.py", line 124, in run_evaluation
    self.load_artifacts()
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_7912\376599490.py", line 15, in load_artifacts
    self.sim_matrix = pickle.load(f)
                      ^^^^^^^^^^^^^^
MemoryError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2201, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^

In [29]:
import pandas as pd 
df = pd.read_csv(r'artifacts\data_transformation\transformed data\Transformed_Data.csv')

In [15]:
import pickle

with open(r'artifacts\feature_engineering\model\sim_matrix.pkl', 'rb') as f:
    sim_matrix = pickle.load(f)
    sim_matrix

In [20]:
sim_matrix[1]

array([0.15691217, 1.        , 0.09687231, ..., 0.        , 0.        ,
       0.        ], shape=(13813,))

In [27]:
sim_matrix[54].argsort()[::-1][0:10]

array([  54, 2114, 8453, 2172,  106, 8440,  163, 8580,  258, 8539])

In [73]:
df[(df["aesthetic"] == 'streetwear')].index[1::][1:10]

Index([2, 3, 4, 5, 6, 7, 8, 9, 10], dtype='int64')

In [72]:
sim_matrix[1].argsort()[::-1][1:10]

array([8421, 2095,    1,   43, 8428,    5, 8426, 7354, 8435])

In [65]:
relevant_idx = df[
    (df["aesthetic"] == aesthetic) &
    (df.index != product_id)
].index.tolist()

In [66]:
print("relevant_idx count :", len(relevant_idx))
print("first 10           :", relevant_idx[:10])

relevant_idx count : 2581
first 10           : [3946, 3947, 3948, 3949, 3950, 3951, 3952, 3953, 3954, 3955]


In [59]:
df.loc[2000]

asin                                                     B0D1XXW98D
product_name      Unisex Regular Cotton Combo of 3 Solid Color T...
price                                                      6.809039
rating                                                          4.2
review_count                                               6.888572
discount                                                         55
image_url         https://m.media-amazon.com/images/I/41ITY79hWU...
product_link      https://www.amazon.in/XENOVAURBAN-Unisex-Regul...
aesthetic                                            college_casual
is_new_product                                                    0
Name: 2000, dtype: object